# RAG System — SubFlo A8

**Knowledge Base:** Spotify Legal Documents (`Spotify_legal.txt`)  
**Sections:** Terms of Use · Intellectual Property Policy · User Guidelines · Paid Subscription Terms

---

## Part 1: Build the RAG Pipeline

### Step 1.1: Install & Import Dependencies

In [ ]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
# !pip install sentence-transformers transformers torch accelerate scikit-learn numpy bitsandbytes>=0.46.1

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import os
import re
import time
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All imports successful.")

All imports successful.


---
### Step 1.2: Load Knowledge Base & Chunking

**Dataset:** `data/Spotify_legal.txt`  
Sections are separated by `---` markers.

**How paragraphs are defined:**  
A paragraph is a non-empty block of text separated by one or more blank lines within a section.

**Three chunking strategies implemented:**
1. **Fixed-Length Chunking** — each chunk is a fixed number of words (~150 words), never splitting mid-sentence.
2. **Overlapping Paragraph Chunking** — adjacent paragraphs merged with 1-paragraph overlap.
3. **Hybrid/Strategic Chunking** — each legal section is kept as its own semantic unit; within each section, paragraphs are grouped by topic heading proximity.

In [ ]:
# =========================================================
# LOAD KNOWLEDGE BASE
# =========================================================

# Relative path from notebook location to the data file
DATA_PATH = Path("Spotify_legal.txt")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Loaded {len(raw_text):,} characters from {DATA_PATH.name}")

Loaded 96,620 characters from Spotify_legal.txt


In [ ]:
# =========================================================
# PARSE INTO SECTIONS AND PARAGRAPHS
# =========================================================

# Split into sections by the '---' separator
sections = [s.strip() for s in raw_text.split("---") if s.strip()]
print(f"Number of sections: {len(sections)}")

# Parse each section into paragraphs (blocks separated by blank lines)
def parse_paragraphs(section_text):
    """Split a section into non-empty paragraph blocks."""
    blocks = re.split(r"\n{2,}", section_text)
    paragraphs = []
    for block in blocks:
        cleaned = " ".join(block.split())  # normalize whitespace
        if len(cleaned.split()) >= 10:     # skip very short blocks
            paragraphs.append(cleaned)
    return paragraphs

all_paragraphs = []
for sec in sections:
    all_paragraphs.extend(parse_paragraphs(sec))

print(f"Total paragraphs parsed: {len(all_paragraphs)}")
total_words = sum(len(p.split()) for p in all_paragraphs)
print(f"Total words: {total_words:,}")
print(f"\nSample paragraph (index 0):\n{all_paragraphs[0][:300]}...")

Number of sections: 4
Total paragraphs parsed: 211
Total words: 15,122

Sample paragraph (index 0):
1. Introduction Please read these Terms of Use ("Terms") carefully as they govern your use of (which includes access to) Spotify's personalized services for streaming music and other content, including all of our websites and software applications that incorporate or link to these Terms (collectivel...


In [ ]:
# =========================================================
# CHUNKING STRATEGY 1: FIXED-LENGTH CHUNKING
# =========================================================
# Chunks the full document at fixed word count (~150 words).
# We do NOT split in the middle of a sentence.

FIXED_CHUNK_WORDS = 150

def fixed_length_chunks(paragraphs, target_words=FIXED_CHUNK_WORDS):
    """Merge paragraphs into chunks of roughly target_words words."""
    chunks = []
    current_chunk_words = []

    for para in paragraphs:
        words = para.split()
        current_chunk_words.extend(words)

        if len(current_chunk_words) >= target_words:
            chunks.append(" ".join(current_chunk_words))
            current_chunk_words = []

    # Append any remaining words as a final chunk
    if current_chunk_words:
        chunks.append(" ".join(current_chunk_words))

    return chunks

chunks_fixed = fixed_length_chunks(all_paragraphs)

print(f"Fixed-Length Chunks: {len(chunks_fixed)}")
avg_words = np.mean([len(c.split()) for c in chunks_fixed])
print(f"Average chunk length: {avg_words:.0f} words")
print(f"\nSample (chunk 0, first 300 chars):\n{chunks_fixed[0][:300]}...")

Fixed-Length Chunks: 72
Average chunk length: 210 words

Sample (chunk 0, first 300 chars):
1. Introduction Please read these Terms of Use ("Terms") carefully as they govern your use of (which includes access to) Spotify's personalized services for streaming music and other content, including all of our websites and software applications that incorporate or link to these Terms (collectivel...


In [ ]:
# =========================================================
# CHUNKING STRATEGY 2: OVERLAPPING PARAGRAPH CHUNKING
# =========================================================
# Each chunk = paragraph[i] + paragraph[i+1].
# Adjacent chunks share one paragraph of overlap.
#
# Example:
#   Chunk 0 -> Para 0 + Para 1
#   Chunk 1 -> Para 1 + Para 2
#   Chunk 2 -> Para 2 + Para 3

def overlapping_paragraph_chunks(paragraphs):
    """Combine adjacent paragraphs with 1-paragraph overlap."""
    chunks = []
    for i in range(len(paragraphs) - 1):
        chunk = paragraphs[i] + " " + paragraphs[i + 1]
        chunks.append(chunk)
    # Also add the very last paragraph alone if not empty
    if paragraphs:
        chunks.append(paragraphs[-1])
    return chunks

chunks_overlap = overlapping_paragraph_chunks(all_paragraphs)

print(f"Overlapping Paragraph Chunks: {len(chunks_overlap)}")
avg_words = np.mean([len(c.split()) for c in chunks_overlap])
print(f"Average chunk length: {avg_words:.0f} words")
print(f"\nSample (chunk 0, first 300 chars):\n{chunks_overlap[0][:300]}...")

Overlapping Paragraph Chunks: 211
Average chunk length: 143 words

Sample (chunk 0, first 300 chars):
1. Introduction Please read these Terms of Use ("Terms") carefully as they govern your use of (which includes access to) Spotify's personalized services for streaming music and other content, including all of our websites and software applications that incorporate or link to these Terms (collectivel...


In [ ]:
# =========================================================
# CHUNKING STRATEGY 3: HYBRID / STRATEGIC CHUNKING
# =========================================================
# Strategy: Treat each top-level legal SECTION as a separate context boundary.
# Within each section, group consecutive paragraphs under the same
# numbered heading (e.g. "1.", "2.", "2.1") into one chunk.
#
# Rationale: Legal documents have strong heading-based structure.
# Grouping by section/subsection keeps related legal clauses together
# and avoids mixing context from unrelated sections.

def hybrid_chunks(sections_text):
    """Group paragraphs by legal section heading within each document section."""
    heading_pattern = re.compile(r"^(\d+(\.\d+)*\.?\s)")
    chunks = []

    for sec in sections_text:
        paragraphs = parse_paragraphs(sec)
        current_group = []

        for para in paragraphs:
            is_new_heading = bool(heading_pattern.match(para))

            if is_new_heading and current_group:
                # Flush current group as one chunk
                chunks.append(" ".join(current_group))
                current_group = []

            current_group.append(para)

        # Flush remaining paragraphs
        if current_group:
            chunks.append(" ".join(current_group))

    return chunks

chunks_hybrid = hybrid_chunks(sections)

print(f"Hybrid Chunks: {len(chunks_hybrid)}")
avg_words = np.mean([len(c.split()) for c in chunks_hybrid])
print(f"Average chunk length: {avg_words:.0f} words")
print(f"\nSample (chunk 0, first 300 chars):\n{chunks_hybrid[0][:300]}...")

Hybrid Chunks: 29
Average chunk length: 521 words

Sample (chunk 0, first 300 chars):
1. Introduction Please read these Terms of Use ("Terms") carefully as they govern your use of (which includes access to) Spotify's personalized services for streaming music and other content, including all of our websites and software applications that incorporate or link to these Terms (collectivel...


---
### Step 1.3: Embedding Pipeline

We use **3 HuggingFace sentence-transformers** of different sizes:

| Size | Model | Dimensions |
|------|-------|------------|
| Small | `all-MiniLM-L6-v2` | 384 |
| Medium | `all-mpnet-base-v2` | 768 |
| Large | `BAAI/bge-large-en-v1.5` | 1024 |

In [ ]:
# =========================================================
# LOAD EMBEDDING MODELS
# =========================================================

EMBEDDING_MODELS = {
    "small":  "all-MiniLM-L6-v2",       # 384 dims
    "medium": "all-mpnet-base-v2",       # 768 dims
    "large":  "BAAI/bge-large-en-v1.5",  # 1024 dims
}

loaded_models = {}

for size_label, model_name in EMBEDDING_MODELS.items():
    print(f"Loading {size_label} model: {model_name} ...")
    start = time.perf_counter()
    loaded_models[size_label] = SentenceTransformer(model_name)
    elapsed = time.perf_counter() - start
    dim = loaded_models[size_label].get_sentence_embedding_dimension()
    print(f"  -> Loaded in {elapsed:.1f}s | Embedding dim: {dim}")

print("\nAll models loaded.")

Loading small model: all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> Loaded in 2.8s | Embedding dim: 384
Loading medium model: all-mpnet-base-v2 ...


...

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> Loaded in 2.5s | Embedding dim: 768
Loading large model: BAAI/bge-large-en-v1.5 ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> Loaded in 6.3s | Embedding dim: 1024

All models loaded.


In [ ]:
# =========================================================
# EMBED ALL CHUNKS FOR ALL MODEL × STRATEGY COMBINATIONS
# =========================================================
# We store embeddings in a nested dict:
#   embeddings_store[model_size][strategy] = numpy array of shape (N, dim)

CHUNKS = {
    "fixed":    chunks_fixed,
    "overlap":  chunks_overlap,
    "hybrid":   chunks_hybrid,
}

embeddings_store = {}

for model_size, model in loaded_models.items():
    embeddings_store[model_size] = {}
    for strategy, chunks in CHUNKS.items():
        print(f"Embedding [{model_size}] x [{strategy}] ({len(chunks)} chunks) ...")
        start = time.perf_counter()
        emb = model.encode(chunks, show_progress_bar=False)
        elapsed = time.perf_counter() - start
        embeddings_store[model_size][strategy] = emb
        print(f"  -> Shape: {emb.shape}  |  Time: {elapsed:.2f}s")

print("\nAll embeddings built.")

Embedding [small] x [fixed] (72 chunks) ...
  -> Shape: (72, 384)  |  Time: 1.45s
Embedding [small] x [overlap] (211 chunks) ...
  -> Shape: (211, 384)  |  Time: 0.72s
Embedding [small] x [hybrid] (29 chunks) ...
  -> Shape: (29, 384)  |  Time: 0.19s
Embedding [medium] x [fixed] (72 chunks) ...
  -> Shape: (72, 768)  |  Time: 1.54s
Embedding [medium] x [overlap] (211 chunks) ...
  -> Shape: (211, 768)  |  Time: 2.59s
Embedding [medium] x [hybrid] (29 chunks) ...
  -> Shape: (29, 768)  |  Time: 0.75s
Embedding [large] x [fixed] (72 chunks) ...
  -> Shape: (72, 1024)  |  Time: 4.56s
Embedding [large] x [overlap] (211 chunks) ...
  -> Shape: (211, 1024)  |  Time: 8.62s
Embedding [large] x [hybrid] (29 chunks) ...
  -> Shape: (29, 1024)  |  Time: 2.87s

All embeddings built.


---
### Step 1.4: Retrieval System

For a given query, we:
1. Embed the query using the chosen model.
2. Compute cosine similarity against all chunk embeddings.
3. Return the top-k most relevant chunks.

In [ ]:
# =========================================================
# RETRIEVAL FUNCTION
# =========================================================

def retrieve(query, model_size, strategy, top_k=3):
    """
    Embed the query and return the top-k most similar chunks.

    Returns:
        list of (score, chunk_text) tuples, sorted by score descending.
    """
    model = loaded_models[model_size]
    chunk_embeddings = embeddings_store[model_size][strategy]
    chunks = CHUNKS[strategy]

    query_emb = model.encode([query])
    scores = cosine_similarity(query_emb, chunk_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]
    results = [(float(scores[i]), chunks[i]) for i in top_indices]
    return results


# Quick smoke test
test_results = retrieve(
    "How do I cancel my Spotify subscription?",
    model_size="small",
    strategy="fixed",
    top_k=2
)

print("Smoke test — top 2 results:")
for score, chunk in test_results:
    print(f"\n  Score: {score:.4f}")
    print(f"  Chunk (first 200 chars): {chunk[:200]}...")

Smoke test — top 2 results:

  Score: 0.7730
  Chunk (first 200 chars): 3. Payment; Cancellation Unless otherwise indicated (for example, if you have signed up for a Prepaid Period (as defined in the Terms of Use)), Paid Subscriptions continue indefinitely until cancelled...

  Score: 0.6758
  Chunk (first 200 chars): If you have purchased a Paid Subscription using a Code, your subscription will automatically terminate at the end of the period stated with your Code, or when there is an insufficient prepaid balance ...


---
### Step 1.5: Generation Pipeline

We use the **local HuggingFace model** selected for this assignment:
`ibm-granite/granite-3.1-2b-instruct`

The prompt is structured as:
- **System instruction** — role of the assistant
- **Retrieved context** — top-k chunks
- **User query** — the question

In [ ]:
GEN_MODEL_NAME = "ibm-granite/granite-3.1-2b-instruct"

_gen_model = None
_gen_tokenizer = None

def get_gen_model():
    global _gen_model, _gen_tokenizer
    if _gen_model is None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

        print(f"Loading generative model: {GEN_MODEL_NAME} ...")
        start = time.perf_counter()
        _gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
        _gen_model = AutoModelForCausalLM.from_pretrained(
            GEN_MODEL_NAME,
            quantization_config=quantization_config,
            device_map="auto",
        )
        elapsed = time.perf_counter() - start
        print(f"  -> Loaded in {elapsed:.1f}s")
    return _gen_model, _gen_tokenizer

print("Generative model will be loaded on first generate() call.")

Generative model will be loaded on first generate() call.


In [ ]:
# =========================================================
# GENERATION FUNCTION
# =========================================================

def build_prompt(query, retrieved_chunks):
    """Construct the RAG prompt from retrieved context and user query."""
    context = "\n\n".join(
        [f"[Context {i+1}]:\n{chunk}" for i, (_, chunk) in enumerate(retrieved_chunks)]
    )
    prompt = (
        "You are a helpful assistant that answers questions about Spotify's "
        "terms, policies, and subscription rules. "
        "Use ONLY the provided context to answer. "
        "If the context does not contain the answer, say 'I don't know based on the provided context.'\n\n"
        f"{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt


def generate_answer(query, model_size, strategy, top_k=3):
    """
    Full RAG pipeline: retrieve + generate.

    Returns:
        dict with keys: retrieved_chunks, prompt, answer, retrieval_latency, gen_latency
    """
    # --- Retrieval ---
    t0 = time.perf_counter()
    retrieved = retrieve(query, model_size, strategy, top_k=top_k)
    retrieval_latency = time.perf_counter() - t0

    # --- Prompt construction ---
    prompt = build_prompt(query, retrieved)

    # --- Generation ---
    model, tokenizer = get_gen_model()

    messages = [{"role": "user", "content": prompt}]

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = prompt

    inputs = tokenizer([text], return_tensors="pt", padding=True).to(model.device)

    t1 = time.perf_counter()
    output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    gen_latency = time.perf_counter() - t1

    out_ids = output_ids[0][len(inputs.input_ids[0]):]
    answer = tokenizer.decode(out_ids, skip_special_tokens=True).strip()

    return {
        "query": query,
        "model_size": model_size,
        "strategy": strategy,
        "retrieved_chunks": retrieved,
        "prompt": prompt,
        "answer": answer,
        "retrieval_latency": retrieval_latency,
        "gen_latency": gen_latency,
        "total_latency": retrieval_latency + gen_latency,
    }


print("Generation function ready.")

Generation function ready.


In [ ]:
# =========================================================
# GENERATION SMOKE TEST (small model, fixed chunking)
# =========================================================

result = generate_answer(
    query="How do I cancel my Spotify subscription?",
    model_size="small",
    strategy="fixed",
    top_k=3
)

print(f"Query: {result['query']}")
print(f"\nRetrieved Chunks:")
for i, (score, chunk) in enumerate(result["retrieved_chunks"]):
    print(f"  [{i+1}] Score={score:.4f} | {chunk[:150]}...")

print(f"\nGenerated Answer:\n{result['answer']}")
print(f"\nRetrieval latency: {result['retrieval_latency']:.2f}s")
print(f"Generation latency: {result['gen_latency']:.2f}s")
print(f"Total latency: {result['total_latency']:.2f}s")

Loading generative model: ibm-granite/granite-3.1-2b-instruct ...


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

  -> Loaded in 37.6s
Query: How do I cancel my Spotify subscription?

Retrieved Chunks:
  [1] Score=0.7730 | 3. Payment; Cancellation Unless otherwise indicated (for example, if you have signed up for a Prepaid Period (as defined in the Terms of Use)), Paid S...
  [2] Score=0.6758 | If you have purchased a Paid Subscription using a Code, your subscription will automatically terminate at the end of the period stated with your Code,...
  [3] Score=0.6724 | 4.3 Trial duration and cancellation. In the case of any Trial, the corresponding Trial Period shall continue for the period as advertised, subject to ...

Generated Answer:
You can cancel your Spotify subscription by logging into your Spotify account and following the prompts on the Account page, or by clicking here and following the instructions. Cancellation will take effect from the end of the billing period in which you cancel. If you have subscribed to a Paid Subscription that includes an initial trial period, you should read sect

---
## Part 2: System Experiments

We test **3 embedding models × 3 chunking strategies = 9 configurations**  
across **5 realistic test queries**.

### Step 2.1: Define Test Queries

In [ ]:
# =========================================================
# TEST QUERIES
# =========================================================
# 5 realistic queries a Spotify user might ask.

TEST_QUERIES = [
    "How do I cancel my Spotify subscription?",
    "What are the eligibility requirements for a student discount?",
    "Can I share my account with family members?",
    "What happens when my free trial ends?",
    "How does Spotify handle copyright infringement claims?",
]

print("Test queries:")
for i, q in enumerate(TEST_QUERIES, 1):
    print(f"  Q{i}: {q}")

Test queries:
  Q1: How do I cancel my Spotify subscription?
  Q2: What are the eligibility requirements for a student discount?
  Q3: Can I share my account with family members?
  Q4: What happens when my free trial ends?
  Q5: How does Spotify handle copyright infringement claims?


---
### Step 2.2: Run All 9 Configurations × 5 Queries

For each configuration we record:
- Retrieved chunks (with similarity scores)
- Generated answer
- Latency (retrieval + generation)

In [ ]:
# =========================================================
# RUN ALL EXPERIMENTS
# =========================================================
# This cell takes a while (9 configs × 5 queries × generation time).

import torch
import gc # Import garbage collection module

MODEL_SIZES = ["small", "medium", "large"]
STRATEGIES  = ["fixed", "overlap", "hybrid"]
TOP_K = 3



all_results = []

total_runs = len(MODEL_SIZES) * len(STRATEGIES) * len(TEST_QUERIES)
run_idx = 0


for model_size in MODEL_SIZES:
    for strategy in STRATEGIES:
        for query in TEST_QUERIES:
            run_idx += 1


            print(f"[{run_idx}/{total_runs}] model={model_size} | strategy={strategy} | query={query[:50]}...")

            # Adjust TOP_K for memory-intensive configurations
            current_top_k = TOP_K
            if strategy == "hybrid":
                current_top_k = 1 # Reduce context size to prevent OOM

            result = generate_answer(
                query=query,
                model_size=model_size,
                strategy=strategy,
                top_k=current_top_k,
            )
            all_results.append(result)

            print(f"   Total latency: {result['total_latency']:.2f}s")

            all_results_path = Path("rag_experiment_results.json")
            with open(all_results_path, "w", encoding="utf-8") as f:
                json.dump(all_results, f, indent=2)

            # Explicitly delete result and clear CUDA cache to free up memory
            del result
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

print(f"\nAll {len(all_results)} experiments complete.")

[1/45] model=small | strategy=fixed | query=How do I cancel my Spotify subscription?...
   Total latency: 20.14s
[2/45] model=small | strategy=fixed | query=What are the eligibility requirements for a studen...
   Total latency: 20.07s
[3/45] model=small | strategy=fixed | query=Can I share my account with family members?...
   Total latency: 16.39s
[4/45] model=small | strategy=fixed | query=What happens when my free trial ends?...
   Total latency: 16.02s
[5/45] model=small | strategy=fixed | query=How does Spotify handle copyright infringement cla...
   Total latency: 26.05s
[6/45] model=small | strategy=overlap | query=How do I cancel my Spotify subscription?...
   Total latency: 15.70s
[7/45] model=small | strategy=overlap | query=What are the eligibility requirements for a studen...
   Total latency: 18.39s
[8/45] model=small | strategy=overlap | query=Can I share my account with family members?...
   Total latency: 10.55s
[9/45] model=small | strategy=overlap | query=What happen

In [ ]:
all_results_path = Path("rag_experiment_results.json")
with open(all_results_path, "r", encoding="utf-8") as f:
    all_results = json.load(f)

In [ ]:
import json
import csv
from pathlib import Path

# Load JSON
all_results_path = Path("rag_experiment_results.json")
with open(all_results_path, "r", encoding="utf-8") as f:
    all_results = json.load(f)

# CSV output path
csv_path = Path("rag_experiment_results.csv")

# CSV headers
headers = [
    "Embedding Model",
    "Chunking Strategy",
    "Query",
    "Retrieved Chunk 1",
    "Retrieved Chunk 2",
    "Retrieved Chunk 3",
    "Response",
    "Retrieved Context Quality",
    "Answer Quality",
    "Latency",
    "Notes"
]

# Write CSV
with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=headers)
    writer.writeheader()

    for entry in all_results:
        # Take the highest retrieved chunk quality
        retrieved_quality = ""
        if "retrieved_chunks" in entry and entry["retrieved_chunks"]:
            retrieved_quality = max(chunk[0] for chunk in entry["retrieved_chunks"])

        row = {
            "Embedding Model": entry.get("model_size", ""),
            "Chunking Strategy": entry.get("strategy", ""),
            "Query": entry.get("query", ""),
            "Retrieved Chunk 1": entry.get("retrieved_chunks", [("", "")])[0][1] if len(entry.get("retrieved_chunks", [])) > 0 else "",
            "Retrieved Chunk 2": entry.get("retrieved_chunks", [("", "")])[1][1] if len(entry.get("retrieved_chunks", [])) > 1 else "",
            "Retrieved Chunk 3": entry.get("retrieved_chunks", [("", "")])[2][1] if len(entry.get("retrieved_chunks", [])) > 2 else "",
            "Response": entry.get("answer", ""),
            "Retrieved Context Quality": "",  # leave blank
            "Answer Quality": "",  # leave blank
            "Latency": entry.get("total_latency", ""),
            "Notes": ""  # leave blank
        }

        writer.writerow(row)

print(f"CSV file saved to {csv_path}")

CSV file saved to rag_experiment_results.csv


In [ ]:
# =========================================================
# DISPLAY RESULTS TABLE
# =========================================================

print(f"{'Model':<8} {'Strategy':<10} {'Q#':<4} {'Ret.Score(top1)':<18} {'Total Lat(s)':<14} Answer Preview")
print("-" * 110)

for r in all_results:
    q_idx = TEST_QUERIES.index(r["query"]) + 1
    top_score = r["retrieved_chunks"][0][0] if r["retrieved_chunks"] else 0.0
    answer_preview = r["answer"][:60].replace("\n", " ")
    print(f"{r['model_size']:<8} {r['strategy']:<10} Q{q_idx:<3} {top_score:<18.4f} {r['total_latency']:<14.2f} {answer_preview}...")

Model    Strategy   Q#   Ret.Score(top1)    Total Lat(s)   Answer Preview
--------------------------------------------------------------------------------------------------------------
small    fixed      Q1   0.7730             20.14          You can cancel your Spotify subscription by logging into you...
small    fixed      Q2   0.6232             20.07          Eligibility for a student discount on Spotify's Paid Subscri...
small    fixed      Q3   0.4428             16.39          Yes, you can share your account with family members by creat...
small    fixed      Q4   0.6236             16.02          When your free trial ends, you will lose access to the Paid ...
small    fixed      Q5   0.7277             26.05          Spotify reviews claims of copyright infringement received th...
small    overlap    Q1   0.8457             15.70          You can cancel your Spotify Paid Subscription by logging int...
small    overlap    Q2   0.6556             18.39          Eligibility for a 

In [ ]:
# =========================================================
# SHOW RETRIEVED CHUNKS FOR EACH RESULT
# =========================================================

for r in all_results:
    q_idx = TEST_QUERIES.index(r["query"]) + 1
    print("=" * 80)
    print(f"Model: {r['model_size']}  |  Strategy: {r['strategy']}  |  Q{q_idx}: {r['query']}")
    print("-" * 80)
    print("RETRIEVED CHUNKS:")
    for i, (score, chunk) in enumerate(r["retrieved_chunks"]):
        print(f"  [{i+1}] Score={score:.4f}")
        print(f"       {chunk[:250]}...")
    print(f"\nGENERATED ANSWER:\n{r['answer']}")
    print(f"\nRetrieval: {r['retrieval_latency']:.2f}s  |  Generation: {r['gen_latency']:.2f}s  |  Total: {r['total_latency']:.2f}s")
    print()

Model: small  |  Strategy: fixed  |  Q1: How do I cancel my Spotify subscription?
--------------------------------------------------------------------------------
RETRIEVED CHUNKS:
  [1] Score=0.7730
       3. Payment; Cancellation Unless otherwise indicated (for example, if you have signed up for a Prepaid Period (as defined in the Terms of Use)), Paid Subscriptions continue indefinitely until cancelled. You will be billed on a recurring basis on the f...
  [2] Score=0.6758
       If you have purchased a Paid Subscription using a Code, your subscription will automatically terminate at the end of the period stated with your Code, or when there is an insufficient prepaid balance to pay for the Spotify Service. In addition to, an...
  [3] Score=0.6724
       4.3 Trial duration and cancellation. In the case of any Trial, the corresponding Trial Period shall continue for the period as advertised, subject to section 4.2, above. Unless cancelled before the end of the Trial Period, or such Tr

---
### Step 2.3: Compare Embedding Models

**Question:** How does embedding size affect retrieval quality and answer quality?  
Did larger embeddings always perform better?

In [ ]:
# =========================================================
# COMPARE EMBEDDING MODELS — Average top-1 retrieval score
# =========================================================

print("Average top-1 retrieval score by embedding model (across all queries and strategies):\n")

for model_size in MODEL_SIZES:
    model_results = [r for r in all_results if r["model_size"] == model_size]
    avg_score = np.mean([r["retrieved_chunks"][0][0] for r in model_results])
    avg_lat = np.mean([r["total_latency"] for r in model_results])
    dim = loaded_models[model_size].get_sentence_embedding_dimension()
    print(f"  {model_size:8s} ({dim} dims) | Avg top-1 score: {avg_score:.4f} | Avg total latency: {avg_lat:.2f}s")

Average top-1 retrieval score by embedding model (across all queries and strategies):

  small    (384 dims) | Avg top-1 score: 0.6490 | Avg total latency: 15.35s
  medium   (768 dims) | Avg top-1 score: 0.6520 | Avg total latency: 14.12s
  large    (1024 dims) | Avg top-1 score: 0.7715 | Avg total latency: 15.05s


Analysis:
- The Average Retrieved Scores for small, medium, and large embedding models are $7.40$, $6.87$, and $7.47$, respectively.
- The Average Answer Scores for small, medium, and large embedding models are $6.63$, $6.00$, and $6.50$, respectively.
- Based on our experiment, we would not conclude that larger embeddings always perform better than their smaller counterparts. For example, the average answer score of our large model is lower than that of our small model. However, it is possible that large embedding models perform better if more experiments are conducted, as both the average retrieved score and answer score of our large model are higher than those of the medium model.

---
### Step 2.4: Compare Chunking Strategies

**Question:** Which chunking strategy worked better? How did chunking affect retrieval relevance and final answers?

In [ ]:
# =========================================================
# COMPARE CHUNKING STRATEGIES — Average top-1 retrieval score
# =========================================================

print("Average top-1 retrieval score by chunking strategy (across all queries and models):\n")

for strategy in STRATEGIES:
    strat_results = [r for r in all_results if r["strategy"] == strategy]
    avg_score = np.mean([r["retrieved_chunks"][0][0] for r in strat_results])
    avg_chunks = np.mean([len(CHUNKS[strategy])])
    print(f"  {strategy:10s} | Num chunks: {len(CHUNKS[strategy]):4d} | Avg top-1 score: {avg_score:.4f}")

Average top-1 retrieval score by chunking strategy (across all queries and models):

  fixed      | Num chunks:   72 | Avg top-1 score: 0.6832
  overlap    | Num chunks:  211 | Avg top-1 score: 0.7168
  hybrid     | Num chunks:   29 | Avg top-1 score: 0.6725


Analysis:
- The Average Retrieved Scores for fixed, overlap, and hybrid chunking strategy are $7.53$, $7.06$, and $7.13$, respectively.
- The Average Answer Scores for fixed, overlap, and hybrid chunking strategy are $6.70$, $6.07$, and $6.37$, respectively.

- Based on our experiment, the fixed chunking strategy gives the best scores on both retrieved quality and answer quality, followed by hybrid and overlap, respectively. Fixed-length chunking likely performs best because it creates consistently sized, self-contained units that capture complete ideas without being too large, maintaining high retrieval relevance. Overlapping chunks add redundancy that can confuse the retriever, while hybrid chunks vary widely in size—too large chunks include irrelevant context, too small chunks fragment information—both reducing retrieval and answer quality. Fixed-length hits a sweet spot between context completeness and focus.

---
### Step 2.5: Data Scaling Experiment

We test with a **smaller subset** (first 25% of paragraphs) and the **full dataset** to see how dataset size affects retrieval quality and noise.

In [ ]:
# =========================================================
# DATA SCALING EXPERIMENT
# =========================================================
# We use the small embedding model and fixed chunking for simplicity.

SCALE_MODEL = "small"
SCALE_QUERY = "How do I cancel my Spotify subscription?"

# Smaller subset: first 25% of paragraphs
subset_paragraphs = all_paragraphs[: len(all_paragraphs) // 4]
chunks_small_scale = fixed_length_chunks(subset_paragraphs)

# Full dataset (already computed)
chunks_full_scale = chunks_fixed

# Embed both
model_small = loaded_models[SCALE_MODEL]
emb_small_scale = model_small.encode(chunks_small_scale)
emb_full_scale  = embeddings_store[SCALE_MODEL]["fixed"]

def retrieve_custom(query, model, chunk_embeddings, chunks, top_k=3):
    query_emb = model.encode([query])
    scores = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(float(scores[i]), chunks[i]) for i in top_indices]

print("=" * 60)
print(f"Query: {SCALE_QUERY}")
print("=" * 60)

for label, emb, chunks_set in [
    ("SMALL SUBSET", emb_small_scale, chunks_small_scale),
    ("FULL DATASET", emb_full_scale,  chunks_full_scale),
]:
    results = retrieve_custom(SCALE_QUERY, model_small, emb, chunks_set, top_k=3)
    print(f"\n--- {label} ({len(chunks_set)} chunks) ---")
    for i, (score, chunk) in enumerate(results):
        print(f"  [{i+1}] Score={score:.4f} | {chunk[:200]}...")


Query: How do I cancel my Spotify subscription?

--- SMALL SUBSET (17 chunks) ---
  [1] Score=0.6758 | If you have purchased a Paid Subscription using a Code, your subscription will automatically terminate at the end of the period stated with your Code, or when there is an insufficient prepaid balance ...
  [2] Score=0.6667 | The price of a Paid Subscription or Trial is specific to the version of the Spotify Service offered in your country of residence and you agree not to engage in any means to Access the Paid Subscriptio...
  [3] Score=0.6045 | If you have prepaid fees directly to Spotify for a Paid Subscription that Spotify permanently discontinues prior to the end of your Prepaid Period (as that term is defined in the "Payments and cancell...

--- FULL DATASET (72 chunks) ---
  [1] Score=0.7730 | 3. Payment; Cancellation Unless otherwise indicated (for example, if you have signed up for a Prepaid Period (as defined in the Terms of Use)), Paid Subscriptions continue indefinitely unt

Analysis:
- With a smaller dataset, the retrieval pool is limited. Some relevant chunks
  may not exist in the subset, forcing the system to return less relevant results.
- With the full dataset, the system has access to all legal clauses, including
  specific cancellation sections that appear later in the document.
- However, a larger dataset also increases the risk of retrieving slightly
  off-topic chunks (noise), especially with weaker embedding models.